In [186]:
import random
import numpy as np

from scipy.special import softmax






reward_1 = 100
reward_6 = 40

def step_left(state):
  if(state!=1):
    return state-1
  else:
    return -1

def step_right(state):
  if(state!=6):
    return state+1
  else:
    return -1

def step_null(state):
  return state

# print("\n\n")
# for i in range(1,7):
#   print(f" {step_left(i)} <- state {i} ")
#   print(f"state {i} -> {step_right(i)} ")
# print("\n\n")

def new_state(state,action):
  if(action == "L"):
    return step_left(state)
  elif(action == "R"):
    return step_right(state)
  else:
    return step_null(state)

def reward(state):
  if(state==1):
    return reward_1
  elif(state==6):
    return reward_6
  else:
    return 0


#Finds a trajectory from starting state, using return to calculate policy. 
#When returns is None,policy is calculated to be even distribution for actions, 1/3 for each since there are 3 actions
#when given a returns dictionary, it passes that dictionary to decide policy
#it doesnt calculate any return or even reward
#it can (and does), however, use updated policy to find optimal paths
#doesnt have its own returns, it takes return into account when given
def sampler(state,returns=None):
  trajectory = []
  s_init = state
  count = 0
  

  while(s_init!=1 and s_init!=6):

    if returns is None:
      action = random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init),k=1)[0]
    
    else:
      action = random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init,returns),k=1)[0]

    #print(f"current state: {s_init} action:{action} new state: {new_state(s_init,action)} count: {count} policy dist: {Policy.policy_dist_static(s_init)}")
    #print(f"{[s_init,action]}")
    trajectory.append([s_init,action,0,0])
    s_init = new_state(s_init,action)
    count = count + 1
    if((s_init!=6 and s_init!=1) and count>20000):
      return None

  trajectory.append([s_init,random.choices(["L","0","R"],weights=Policy.policy_dist_static(s_init),k=1)[0],0,0])
  
  return trajectory


#given a trajectory, calculate Gt for each state
def calculate_return(trajectory,discount=0.5):

  for i in range(len(trajectory) - 1, -1, -1):
    if(trajectory[i][0]==6 or trajectory[i][0]==1):
      trajectory[i][2] = reward(trajectory[i][0])
      trajectory[i][3] = trajectory[i][2]
    else:
      trajectory[i][3] = reward(trajectory[i][0]) + discount * trajectory[i+1][3]

  return trajectory

  #returned trajectory looks like [s,"action",reward,return]


#finds an average return value for each (s,a) combination
#This is the function that drives the learning.
# this function creates bunch of episodes and keeps track of the moving average of all the possible (s,a) combination.
#this returns a "returns" dictionary which later is used to decide policy function.

def return_estimation(start_state, num_episodes,rets=None,discount=0.5,show=False):
    returns = {}
    counts = {}

    for i in range(num_episodes):
      if rets is None:
        #("we are here")
        s0 = random.randint(1, 6)
        trajectory = sampler(s0)
        #(trajectory)
      else:
        trajectory = sampler(start_state,rets)
      if show:
         print([t[0] for t in trajectory]) 
      if trajectory is not None:
        #print(trajectory)
        trajectory = calculate_return(trajectory,discount=discount)
        for state, action, r, Gt in trajectory:
            key = (state, action)
            if key in returns:
                counts[key] += 1
                # update running average
                returns[key] += (Gt - returns[key]) / counts[key]
            else:
                returns[key] = Gt
                counts[key] = 1

    return returns




class Policy:
    zero_ret = {}
    def __init__(self,init_val=0):
        self.returns = {}
        self.policy_probability ={}
        a = ["L","0","R"]
        for i in range(1,7):
            for j in a:
                key = (i,j)
                self.returns[key] = init_val

        Policy.zero_ret = dict(self.returns) 
        for i in range(1,7):
            self.policy_probability[i] = self.policy_dist(i)

    def policy_dist(self,s,returns=None,):
        if(s==1 or s==6):
          return [0,1,0]
        dist =[]
        if returns == None:
          returns = self.returns
        
        Q1 = returns.get((s,"L"),0)
        Q2 = returns.get((s,"0"),0)
        Q3 = returns.get((s,"R"),0)
        Qv = [Q1,Q2,Q3]
        dist = softmax(Qv) 
            
        return dist
    @staticmethod
    def policy_dist_static(s,returns=None):
        if(s==1 or s==6):
          return [0,1,0]
        dist =[]
        if returns == None:
          returns = Policy.zero_ret
        
        Q1 = returns.get((s,"L"),0)
        Q2 = returns.get((s,"0"),0)
        Q3 = returns.get((s,"R"),0)
        Qv = [Q1,Q2,Q3]
        dist = softmax(Qv) 
            
        return dist
    
    def update(self,returns):
       self.returns = returns
       for i in range(1,7):
            self.policy_probability[i] = self.policy_dist(i)
    
    def update2(self,returns):
      returns_new = dict(self.returns)  # Start with a copy of old returns
      
      for (s,a), r_new in sorted(returns.items()):
        r_old = self.returns.get((s,a), 0)  # Default to 0 if not in old returns
        
        # Exponential moving average with alpha=0.9
        r_updated = r_old + (r_new - r_old) * 0.9
        returns_new[(s,a)] = r_updated

      self.returns = returns_new
      
      for i in range(1,7):
          self.policy_probability[i] = self.policy_dist(i)








In [187]:
returns = return_estimation(4,4000,discount=0.3)

for(s,a),er in sorted(returns.items()):

    print(f" Q: {s}  {a}  -> {er:.4f}\n")

 Q: 1  0  -> 100.0000

 Q: 2  0  -> 3.3407

 Q: 2  L  -> 30.0000

 Q: 2  R  -> 0.3865

 Q: 3  0  -> 0.4473

 Q: 3  L  -> 3.4259

 Q: 3  R  -> 0.1916

 Q: 4  0  -> 0.1951

 Q: 4  L  -> 0.3918

 Q: 4  R  -> 1.3452

 Q: 5  0  -> 1.4328

 Q: 5  L  -> 0.1957

 Q: 5  R  -> 12.0000

 Q: 6  0  -> 40.0000



In [188]:






def naive_RL(iterations=100,discount=0.5,init_state=3):
  
  policyy = Policy()
  print(f"Starting naive RL with discount value:{discount} initial_state:{init_state}")

  print("state:--- policy distribution:  L  0  R BEFORE ITERATION")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policyy.policy_dist(i)[0]}  {policyy.policy_dist(i)[1]}  {policyy.policy_dist(i)[2]}")


  returns = policyy.returns
  for i in range(iterations):
    
    returns = return_estimation(init_state, 4000,returns,discount=discount)

    policyy.update2(returns)

  
  print(f"state:--- policy distribution:  L  0  R after all Iterations ")
  for i in range(1,7):
    print(f" {i}        policy distribution:  {policyy.policy_dist(i)[0]}  {policyy.policy_dist(i)[1]}  {policyy.policy_dist(i)[2]}")

  print("\n\n\n")
  
  return policyy





In [189]:




#THIS IS OUR OPTIMAL POLICY, or rather OUR DISTRIBUTION

policy = Policy()
policy = naive_RL(10,discount=0.5,init_state=4)


print("After dust is settled")

print("Returns for discount 0.5")
for i in range(1,7):
    print(f" {i}        policy distribution:  {policy.policy_dist(i)[0]:0.4f}  {policy.policy_dist(i)[1]:0.4f}  {policy.policy_dist(i)[2]:0.4f}")


  
  

Starting naive RL with discount value:0.5 initial_state:4
state:--- policy distribution:  L  0  R BEFORE ITERATION
 1        policy distribution:  0  1  0
 2        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 3        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 4        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 5        policy distribution:  0.3333333333333333  0.3333333333333333  0.3333333333333333
 6        policy distribution:  0  1  0
state:--- policy distribution:  L  0  R after all Iterations 
 1        policy distribution:  0  1  0
 2        policy distribution:  1.0  2.3506091521614154e-18  1.5660803614146768e-21
 3        policy distribution:  0.9999999998421918  1.1137274894271796e-10  4.643561844161637e-11
 4        policy distribution:  0.9225022235522223  0.0017741823680047478  0.07572359407977311
 5        policy distribution:  6.725911086167021e-09  9.12870

In [190]:
track = []
for i in range(1000):
  tt = sampler(4,policy.returns)
  tt = [row[0] for row in tt]
  track.append(tt)


from collections import Counter
# Convert inner lists to tuples
track_tuples = [tuple(x) for x in track]
# Count occurrences
counts = Counter(track_tuples)
# Total number of elements
total = len(track)
# Calculate percentages
percentages = {key: (value / total) * 100 for key, value in counts.items()}
# Sort percentages by value descending and take top 3
top3 = sorted(percentages.items(), key=lambda x: x[1], reverse=True)[:]
# Print nicely
for element, pct in top3:
    print(f"{list(element)}: {pct:.2f}%")  # convert back to list if needed


[4, 3, 2, 1]: 92.90%
[4, 5, 6]: 6.90%
[4, 4, 3, 2, 1]: 0.20%


In [191]:
import numpy as np

values = np.array([12.5,10])
probs = np.exp(values) / np.sum(np.exp(values))
print(probs)




[0.92414182 0.07585818]
